In [35]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq

In [37]:
import os
from dotenv import load_dotenv
# 1. Load the keys from .env
load_dotenv()
# 2. Override the URL and turn on debug mode
os.environ["LANGFUSE_BASE_URL"] = "http://localhost:3001"
os.environ["LANGFUSE_DEBUG"] = "true"
# 3. Initialize the Langfuse CallbackHandler
from langfuse.langchain import CallbackHandler
langfuse_handler = CallbackHandler()
thread_id = "1"
config = {
    "configurable": {"thread_id": thread_id},
    "callbacks": [langfuse_handler]
}

In [38]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


checkpointer = MemorySaver()

In [39]:
# llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
llm = ChatGroq(model="llama-3.1-8b-instant"
)

def chat_node(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': [response]}

In [40]:
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

# chatbot = graph.compile(checkpointer=checkpointer)
chatbot = graph.compile()

# initial_state = {'messages': [HumanMessage(content="What is the capital of Nepal")]}

# chatbot.invoke(initial_state)

In [41]:
thread_id = "1"

config = {"configurable": {"thread_id": thread_id}}

In [45]:
while True:
    user_message = input('Type here: ')
    print(f'User: {user_message}')

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    print("AI: ", end=" ")

    for message_chunk, metadata in chatbot.stream({'messages': [HumanMessage(content=user_message)]}, stream_mode="messages", config=config):
        if message_chunk.content:
            print(message_chunk.content, end=" ", flush=True)

    # 1. Check if the keys and URL are actually working
    if not langfuse_handler.auth_check():
        print("\n[ERROR] Langfuse connection failed! Check your API keys and URL.")
    else:
        # 2. Get the direct URL to this exact trace!
        trace_url = langfuse_handler.get_trace_url()
        print(f"\n[Trace Link]: {trace_url}")
        
    # 3. Force the flush
    langfuse_handler._langfuse_client.flush()



User: hi my name is prajin
AI:  Hi  Pra jin .  It 's  nice  to  meet  you .  Is  there  something  I  can  help  you  with  or  would  you  like  to  chat ? 

AttributeError: 'LangchainCallbackHandler' object has no attribute 'auth_check'